[Home](../../README.md)

### Data Wrangling

I will be using this notebook for data wrangling through [Pandas](https://pandas.pydata.org/). I will be analysing my data and manipulating it based on the analysis. My final wrangled data will be saved at the end to a new csv file, ready for the next stage in my model development, feature engineering.

In [ ]:
# Import frameworks
import pandas as pd

In [ ]:
data_frame = pd.read_csv("2.1.2.minecraft_100x100.csv")

#### Checking for null values

I'm using `isnull().sum()` to check for any null values throughout my database, as null values can cause runtime errors and skew results.

In [ ]:
pd.set_option("display.max_rows", None)
data_frame.isnull().sum()

#### Replace data

In order to improve the readibility and optimization of my targets and features, I am going to remove the common prefix "minecraft:" from each column.

In [ ]:
data_frame['dominant_biome'] = data_frame['dominant_biome'].apply(lambda x: x.replace('minecraft:', ''))
data_frame['dominant_biome'].head()

In [ ]:
data_frame.columns = data_frame.columns.str.replace("minecraft:", "", regex=False)
data_frame.head()

#### Drop/combine unecessary targets

I'm analysing the different targets, and deciding to combine certain ones based on my domain knowledge. This is to simplify the data for the model and reduce possible overfitting.

In [ ]:
data_frame['dominant_biome'].unique()

In [ ]:
data_frame['dominant_biome'] = data_frame['dominant_biome'].apply(
    lambda biome: 'ocean' if 'ocean' in biome else biome
)
data_frame["dominant_biome"] = data_frame["dominant_biome"].apply(
    lambda biome: "taiga" if "taiga" in biome else biome
)
data_frame["dominant_biome"] = data_frame["dominant_biome"].apply(
    lambda biome: "birch_forest" if "birch_forest" in biome else biome
)
data_frame["dominant_biome"].unique()

#### Drop unecessary features

Many of my features are not necessary for the algorithms prediction, either being very rare (only in one or two chunks), or are in every single chunk no matter what (e.g. bedrock).

In [ ]:
df = pd.read_csv("2.1.2.minecraft_100x100.csv")
zero_counts = (df == 0).sum(axis=0)
zero_summary = pd.DataFrame({"Column": df.columns, "Zero Count": zero_counts.values})
zero_summary = zero_summary.sort_values("Zero Count", ascending=True)
print(zero_summary.to_string(index=False))

I have attempted to filter features by the amount of 0 values to determine their importance, but I have decided that I will need to manually select which features to keep based on my domain knowledge.

In [ ]:
# chunk_x, chunk_z, dominant_biome, air, dirt, water, short_grass, grass_block, lava, sand, tall_seagrass, oak_leaves, birch_leaves, oak_log, tall_grass, birch_log, kelp, kelp_plant, oak_planks, oak_fence, dark_oak_log, dark_oak_leaves, spruce_leaves, spruce_log, bush, sandstone, brown_mushroom, dandelion, podzol, red_mushroom_block, mushroom_stem, poppy, coarse_dirt, oxeye_daisy, cornflower, azure_bluet, emerald_ore, lily_pad, barrel, sugar_cane, red_mushroom, lilac, lily_of_the_valley, rose_bush, peony, sweet_berry_bush, bee_nest, pumpkin, snow, acacia_leaves, acacia_log, blue_orchid

columns_to_keep = [
    "chunk_x",
    "chunk_z",
    "dominant_biome",
    "air",
    "dirt",
    "water",
    "short_grass",
    "grass_block",
    "lava",
    "sand",
    "tall_seagrass",
    "oak_leaves",
    "birch_leaves",
    "oak_log",
    "tall_grass",
    "birch_log",
    "kelp",
    "kelp_plant",
    "oak_planks",
    "oak_fence",
    "dark_oak_log",
    "dark_oak_leaves",
    "spruce_leaves",
    "spruce_log",
    "bush",
    "sandstone",
    "brown_mushroom",
    "dandelion",
    "podzol",
    "red_mushroom_block",
    "mushroom_stem",
    "poppy",
    "coarse_dirt",
    "oxeye_daisy",
    "cornflower",
    "azure_bluet",
    "emerald_ore",
    "lily_pad",
    "barrel",
    "sugar_cane",
    "red_mushroom",
    "lilac",
    "lily_of_the_valley",
    "rose_bush",
    "peony",
    "sweet_berry_bush",
    "bee_nest",
    "pumpkin",
    "snow",
    "acacia_leaves",
    "blue_orchid",
]

data_frame = data_frame[columns_to_keep]
data_frame.head()

#### Remove outliers

My algorithm to remove outliers is fairly complex, as my data requires that I analyse outliers in featuers *per* target, otherwise valid data could be removed. To remove these outliers, I loop through each block through each biome, then find the bottom and top 10% of values, and calculate IQR from that data. Next, we skip any instances where IQR = 0, as this means there is very little to no variation, usually for very rare blocks, and any non-zero value would be flagged as an outlier. This means that valid rows would be removed if this exception wasn't in place. Then we calculate threshholds, and stop the lower at 0, since negative values aren't possible, and then find our outliers. Finally, returning our dataframe missing all rows with outliers

In [ ]:
# get the inter-quartile range per biome for relevant block features
# Outliers are defined relative to each biome's distribution
# When IQR is 0, blocks can be skipped as they are unnecessary to remove
# Algorithm shouldn't be biased from biome overlap, but should still be able to handle occasional misplaced blocks

blocks = [
    "air",
    "dirt",
    "water",
    "short_grass",
    "grass_block",
    "lava",
    "sand",
    "tall_seagrass",
    "oak_leaves",
    "birch_leaves",
    "oak_log",
    "tall_grass",
    "birch_log",
    "kelp",
    "kelp_plant",
    "oak_planks",
    "oak_fence",
    "dark_oak_log",
    "dark_oak_leaves",
    "spruce_leaves",
    "spruce_log",
    "bush",
    "sandstone",
    "brown_mushroom",
    "dandelion",
    "podzol",
    "red_mushroom_block",
    "mushroom_stem",
    "poppy",
    "coarse_dirt",
    "oxeye_daisy",
    "cornflower",
    "azure_bluet",
    "emerald_ore",
    "lily_pad",
    "barrel",
    "sugar_cane",
    "red_mushroom",
    "lilac",
    "lily_of_the_valley",
    "rose_bush",
    "peony",
    "sweet_berry_bush",
    "bee_nest",
    "pumpkin",
    "snow",
    "acacia_leaves",
    "blue_orchid",
]


def filter_biome_outliers(df, blocks):
    mask = pd.Series([True] * len(df), index=df.index)
    for block in blocks:
        for biome, group in df.groupby("dominant_biome"):
            Q1 = group[block].quantile(0.10)
            Q3 = group[block].quantile(0.90)
            IQR = Q3 - Q1
            # Skip sparse blocks where IQR=0 — every non-zero value would be flagged
            if IQR == 0:
                print(f"Skipping {block} in {biome} as IQR = 0")
                continue
            lower = max(0, Q1 - IQR)  # clamp to 0, block counts can't be negative
            upper = Q3 + IQR
            outlier_mask = (df.index.isin(group.index)) & (
                (df[block] < lower) | (df[block] > upper)
            )
            removed = outlier_mask.sum()
            if removed > 0:
                print(
                    f"Removing {removed} outliers for {block} in {biome} (threshold: <{lower:.2f} or >{upper:.2f})"
                )
            mask &= ~outlier_mask
    return df[mask]


rows_before = len(data_frame)
data_frame = filter_biome_outliers(data_frame, blocks)
print(
    f"\nRows before: {rows_before} | Rows after: {len(data_frame)} | Removed: {rows_before - len(data_frame)}"
)

#### Scaling features to a common range

Scaling the features makes it easier for machine learning algorithms to find the optimal solution, as the different scales of the features do not influence them.

In [ ]:
# Record original min/max before scaling
scaling_params = {}
for block in blocks:
    scaling_params[block] = {
        "min": data_frame[block].min(),
        "max": data_frame[block].max(),
    }

# Min-Max scale all block features to [0, 1] range
for block in blocks:
    min_val = scaling_params[block]["min"]
    max_val = scaling_params[block]["max"]
    if max_val - min_val > 0:
        data_frame[block] = (data_frame[block] - min_val) / (max_val - min_val)

# Display scaling params as a readable table for noting in 2.1.2.data.records
scaling_df = pd.DataFrame(scaling_params).T
print(scaling_df.to_string())

#### Correlation heatmap

I'm using a correlation heatmap at the end of my wrangling to find features that have very little correlaton to any target. This is to remove unecessary features and simplify the data so the algorithm can process it faster and find corellation more easily.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Encode dominant_biome as numeric for correlation
biome_encoded = data_frame["dominant_biome"].astype("category").cat.codes

# Compute correlation of each block with each biome (one-hot encoded)
biomes = data_frame["dominant_biome"].unique()
biome_block_corr = pd.DataFrame(index=biomes, columns=blocks, dtype=float)

for biome in biomes:
    # Create binary column: 1 if this biome, 0 otherwise
    biome_indicator = (data_frame["dominant_biome"] == biome).astype(int)
    for block in blocks:
        biome_block_corr.loc[biome, block] = biome_indicator.corr(data_frame[block])

fig, ax = plt.subplots(figsize=(24, 6))

# Plot heatmap using imshow
im = ax.imshow(
    biome_block_corr.values.astype(float),
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    aspect="auto",
)

# Add colourbar
fig.colorbar(im, ax=ax, shrink=0.8)

# Annotate each cell with its correlation value
for i in range(len(biomes)):
    for j in range(len(blocks)):
        val = biome_block_corr.iloc[i, j]
        text_colour = "white" if abs(val) > 0.5 else "black"
        ax.text(
            j, i, f"{val:.2f}", ha="center", va="center", fontsize=6, color=text_colour
        )

# Set tick labels
ax.set_xticks(range(len(blocks)))
ax.set_yticks(range(len(biomes)))
ax.set_xticklabels(blocks, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(biomes, fontsize=9)

ax.set_title("Biome vs Block Feature Correlation Heatmap", fontsize=16)
plt.tight_layout()
plt.show()

Based on my analysis heatmap, I'm doing a 'second round' of removal of certain features, only keeping those with a correlation to any biome above 0.25.

In [ ]:
columns_to_keep = [
    "chunk_x",
    "chunk_z",
    "dominant_biome",
    "air",
    "dirt",
    "water",
    "short_grass",
    "grass_block",
    "sand",
    "tall_seagrass",
    "oak_leaves",
    "birch_leaves",
    "oak_log",
    "tall_grass",
    "birch_log",
    "kelp",
    "kelp_plant",
    "dark_oak_log",
    "dark_oak_leaves",
    "spruce_leaves",
    "spruce_log",
    "bush",
    "sandstone",
    "brown_mushroom",
    "dandelion",
    "podzol",
    "red_mushroom_block",
    "mushroom_stem",
    "poppy",
    "coarse_dirt",
    "oxeye_daisy",
    "emerald_ore",
    "lily_pad",
    "snow",
    "acacia_leaves",
]

data_frame = data_frame[columns_to_keep]
data_frame.head()

#### Save the wrangled data to CSV

In [ ]:
data_frame.to_csv('../2.2.Feature_Engineering/2.2.1.wrangled_data.csv', index=False)